# Notebook → Script certification pipeline (LSTM-AE demo)

This notebook demonstrates the full workflow for certifying an LSTM autoencoder
with **cert-rnn** when the long verification runs must survive overnight on a server:

| Where | What | Why |
|---|---|---|
| **notebook** (this file) | train / load model, pick τ + anchor, convert, `preflight`, `smoke_test`, **export** | interactive, fast steps |
| **script** (`verify_overnight_template.py` in tmux) | the hours-long certification loop | survives VSCode/SSH disconnects, resumable |
| **notebook** again | load `results_overnight.npz` | analyze/plot results |

The model here is a stand-in with the *same structure and naming* as a
Curiousily-style AE (`encoder.rnn1/rnn2`, `decoder.rnn1/rnn2`,
`decoder.output_layer`) at realistic dimensions (T=64, D=1, H=55) —
replace **section 1** with your own model/data and everything else carries over.

In [1]:
# Pin BLAS threads BEFORE importing numpy (cert-rnn does many small ops;
# extra threads only add contention). Harmless if numpy is already loaded.
import os
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import numpy as np
import torch
import torch.nn as nn

from cert_rnn import LSTMAutoencoder, ReconErrorSpec

torch.manual_seed(0)
rng = np.random.default_rng(0)
SEQ_LEN, N_FEATURES, HIDDEN = 64, 1, 55

## 1. Your model + training  *(replace this whole section with yours)*

The only thing the rest of the pipeline cares about is the **submodule naming**
(it determines the `state_dict` keys the script reads) and the **topology**
(latent fed to the decoder at every step, per-step linear head).

In [2]:
class Encoder(nn.Module):
    def __init__(self, n_features, hidden):
        super().__init__()
        self.rnn1 = nn.LSTM(n_features, hidden, batch_first=True)
        self.rnn2 = nn.LSTM(hidden, hidden, batch_first=True)

    def forward(self, x):                    # x: (B, T, D)
        x, _ = self.rnn1(x)
        _, (h, _c) = self.rnn2(x)
        return h[-1]                         # latent: (B, H)


class Decoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden):
        super().__init__()
        self.seq_len = seq_len
        self.rnn1 = nn.LSTM(hidden, hidden, batch_first=True)
        self.rnn2 = nn.LSTM(hidden, hidden, batch_first=True)
        self.output_layer = nn.Linear(hidden, n_features)

    def forward(self, latent):               # latent: (B, H)
        x = latent.unsqueeze(1).repeat(1, self.seq_len, 1)   # (B, T, H)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        return self.output_layer(x)          # (B, T, D)


class RecurrentAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden):
        super().__init__()
        self.encoder = Encoder(n_features, hidden)
        self.decoder = Decoder(seq_len, n_features, hidden)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def make_windows(n):
    """Synthetic 'normal' data: phase-shifted sines, (n, T, D)."""
    t = np.linspace(0, 4 * np.pi, SEQ_LEN)
    X = np.stack([
        np.sin(t + rng.uniform(0, 2 * np.pi))[:, None] * 0.5
        + 0.02 * rng.standard_normal((SEQ_LEN, 1))
        for _ in range(n)
    ])
    return X.astype(np.float32)

model = RecurrentAutoencoder(SEQ_LEN, N_FEATURES, HIDDEN)
X_train, X_val = torch.tensor(make_windows(96)), make_windows(64)

opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for epoch in range(30):
    for i in range(0, len(X_train), 32):
        batch = X_train[i:i + 32]
        opt.zero_grad()
        loss = ((model(batch) - batch) ** 2).mean()
        loss.backward()
        opt.step()
model.eval()
print(f"trained: final batch MSE = {loss.item():.5f}")

trained: final batch MSE = 0.11960


## 2. Convert to cert-rnn (non-mutating) and validate

Note: **no `.double()` anywhere** — `nn.Module.double()` mutates your trained
model in place (that's the Float/Double bug); `from_torch` casts every weight
to float64 internally anyway.

In [3]:
def cell(r):
    """One single-layer nn.LSTM -> equivalent nn.LSTMCell (fresh copy; never mutates)."""
    c = nn.LSTMCell(r.input_size, r.hidden_size).to(r.weight_ih_l0.dtype)
    with torch.no_grad():
        c.weight_ih.copy_(r.weight_ih_l0); c.weight_hh.copy_(r.weight_hh_l0)
        c.bias_ih.copy_(r.bias_ih_l0);     c.bias_hh.copy_(r.bias_hh_l0)
    return c

ae = LSTMAutoencoder.from_torch(
    [cell(model.encoder.rnn1), cell(model.encoder.rnn2)],   # full encoder stack
    [cell(model.decoder.rnn1), cell(model.decoder.rnn2)],   # full decoder stack
    model.decoder.output_layer)                              # as-is, NO .double()

print("captured:", ae.encoder["L"], "enc layers,", ae.decoder["L"], "dec layers",
      " D=", ae.D, " H=", ae.H)

captured: 2 enc layers, 2 dec layers  D= 1  H= 55


In [4]:
# Pick tau + anchor exactly as in your pipeline: score the validation windows,
# set tau at the 95th percentile, anchor = the best-reconstructed window.
scores = ae.score(X_val.astype(np.float64))
tau = float(np.percentile(scores, 95))
i = int(np.argmin(scores))
anchor = X_val[i].astype(np.float64)
print(f"tau={tau:.5f}  anchor#{i}  score={scores[i]:.5f}  (< tau: {scores[i] < tau})")

tau=0.16277  anchor#15  score=0.11589  (< tau: True)


In [5]:
# Gate 1 -- preflight: anchor format vs the model, tau headroom, and
# (crucially) score parity against the ORIGINAL torch model. Run once per model.
report = ae.preflight(anchor, tau=tau, torch_model=model, title="demo AE, best anchor")
print(report)
assert report.ok

Preflight [demo AE, best anchor]: OK -- safe to certify
  checking: model D=1 H=55 L_enc=2 L_dec=2 | anchor (64, 1) | tau=0.162767 | torch parity: yes
  [PASS] dtype: convertible to float64 (source dtype: float64)
  [PASS] shape: (T=64, D=1) matches the model (H=55)
  [INFO] seq length: T=64; the engine accepts any T -- confirm it equals your training window length (cost per reach grows ~T^2)
  [PASS] finite: no NaN/Inf
  [INFO] value range: [-0.5216, 0.5095], mean -0.0002571 -- must be in the same scaling/normalization used at training time
  [INFO] anchor score: 0.115894 (concrete, eps=0)
  [PASS] tau: tau=0.162767 leaves 1.40x headroom over the anchor score
  [PASS] torch parity: tool score 0.115894 == torch score 0.115894 (|diff|=1.86e-09)


In [6]:
# Gate 2 -- smoke test: seconds-fast end-to-end run on a truncated anchor
# + a cost forecast for the real thing. Decide single_frame vs multi_frame here.
smoke = ae.smoke_test(anchor, tau, n_frames=16)
assert smoke["ok"]
print(f"one full-length reach  ~{smoke['est_sec_per_reach_full']:.0f}s")
print(f"certify multi_frame    ~{smoke['est_certify_multi_frame_s'] / 60:.1f} min")
print(f"certify single_frame   ~{smoke['est_certify_single_frame_s'] / 3600:.1f} h  (per-frame profile)")

one full-length reach  ~23s
certify multi_frame    ~5.1 min
certify single_frame   ~5.4 h  (per-frame profile)


## 3. Export — the notebook → script handoff

Two files are the **entire interface** between notebook and script:

- `lstm_ae_checkpoint.pt` — the flat `state_dict` (named weight tensors). The
  script rebuilds `nn.LSTMCell`s straight from the names, so **your model class
  is never needed outside the notebook**.
- `verify_inputs.npz` — the anchor window and τ.


In [7]:
torch.save(model.state_dict(), "lstm_ae_checkpoint.pt")
np.savez("verify_inputs.npz", anchor=anchor, tau=tau)
print("exported:")
print("  lstm_ae_checkpoint.pt  keys:", len(model.state_dict()), "tensors")
print("  verify_inputs.npz      anchor", anchor.shape, " tau", round(tau, 5))

exported:
  lstm_ae_checkpoint.pt  keys: 18 tensors
  verify_inputs.npz      anchor (64, 1)  tau 0.16277


## 4. Run the script (normally: in tmux, overnight)

Copy the template next to this notebook, then launch it **detached** so a
VSCode/SSH disconnect can't kill it:

```bash
tmux new -s verify
python -u verify_overnight_template.py 2>&1 | tee verify.log   # full per-frame run
# detach: Ctrl-b then d        reattach: tmux attach -t verify
```

Monitor from any terminal — the log gains a heartbeat line every ~20 s:

```bash
tail -f verify.log      # live
ls -l verify.log        # mtime > ~2 min old  =>  hung
```

It checkpoints `results_overnight.npz` after **every frame**; rerunning after
a crash skips finished frames automatically.

Below we run it right here (quick options) just to demonstrate the loop —
`--smoke-only` first, then a short `multi_frame` certification.

In [8]:
!cp ../verify_overnight_template.py .
!python -u verify_overnight_template.py --smoke-only

/home/verivital/anaconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
loaded: D=1 H=55 L_enc=2 L_dec=2  anchor (64, 1)  tau=0.162767
Preflight [overnight run]: OK -- safe to certify
  checking: model D=1 H=55 L_enc=2 L_dec=2 | anchor (64, 1) | tau=0.162767
  [PASS] dtype: convertible to float64 (source dtype: float64)
  [PASS] shape: (T=64, D=1) matches the model (H=55)
  [INFO] seq length: T=64; the engine accepts any T -- confirm it equals your training window length (cost per reach grows ~T^2)
  [PASS] finite: no NaN/Inf
  [INFO] value range: [-0.5216, 0.5095], mean -0.0002571 -- must be in the same scaling/normalization used at training time
  [INFO] 

In [9]:
# Quick real certification via the script (multi_frame, ~3 min; eps_init=0.1 suits the data scale).
# Overnight you would instead run the tmux command above (no --n-iters override).
!python -u verify_overnight_template.py --threat-model multi_frame --n-iters 8 --eps-init 0.1

/home/verivital/anaconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
loaded: D=1 H=55 L_enc=2 L_dec=2  anchor (64, 1)  tau=0.162767
Preflight [overnight run]: OK -- safe to certify
  checking: model D=1 H=55 L_enc=2 L_dec=2 | anchor (64, 1) | tau=0.162767
  [PASS] dtype: convertible to float64 (source dtype: float64)
  [PASS] shape: (T=64, D=1) matches the model (H=55)
  [INFO] seq length: T=64; the engine accepts any T -- confirm it equals your training window length (cost per reach grows ~T^2)
  [PASS] finite: no NaN/Inf
  [INFO] value range: [-0.5216, 0.5095], mean -0.0002571 -- must be in the same scaling/normalization used at training time
  [INFO] 

## 5. Back in the notebook: load the results

In [10]:
d = np.load("results_overnight.npz")
print("saved arrays:", list(d.keys()))
if "radii" in d:   # single_frame run: per-frame profile
    radii = d["radii"]
    done = np.isfinite(radii)
    print(f"frames done: {done.sum()}/{len(radii)}")
    print(f"certified radius (min over done frames): {np.nanmin(radii):.6g}")
    print(f"most sensitive frame: {np.nanargmin(radii)}")
else:              # multi_frame run: one joint radius
    print(f"threat_model={d['threat_model']}  certified radius={float(d['radius']):.6g}"
          f"  ({float(d['seconds']):.0f}s)")

saved arrays: ['radius', 'tau', 'threat_model', 'seconds']
threat_model=multi_frame  certified radius=0.0214844  (206s)


## Adapting this to your real notebook — checklist

1. Replace **section 1** with your model/data loading (everything else is generic).
2. If your submodule names differ, edit `ENC_LAYERS`/`DEC_LAYERS`/`HEAD` at the
   top of `verify_overnight_template.py` to match your `state_dict` keys.
3. If your session previously ran an in-place `.double()` on a submodule, run
   `model.float()` once **before** exporting (preflight's parity check will
   catch it if you forget — look for the MIXED-dtype FAIL).
4. Run sections 2–3 (convert → preflight → smoke → export), check the forecast,
   then launch the script in tmux and go home.